<a href="https://colab.research.google.com/github/Zeshanbaloch786/HealthyMe_APP/blob/main/HealthyMe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install groq

In [ ]:
# ----------------------------------
# Install (if running locally)
# pip install gradio pandas groq
# ----------------------------------

import gradio as gr
import pandas as pd
import json
import os
import re
from groq import Groq

# ----------------------------------
# Initialize Groq client
# ----------------------------------
  # Set this in environment
from google.colab import userdata
HealthyMe =userdata.get('HealthyMe')
client = Groq(api_key=HealthyMe)

# ----------------------------------
# Safe JSON Parsing & Backend Logic
# ----------------------------------
def safe_json_parse(text):
    try:
        return json.loads(text)
    except:
        try:
            match = re.search(r"\[.*\]", text, re.DOTALL)
            if match:
                return json.loads(match.group(0))
        except:
            pass
    return []

def generate_ai_diet_plan(calories, protein, fats, carbs, goal, budget):
    if None in [calories, protein, fats, carbs, goal, budget]:
        return gr.update(visible=False), None, None

    prompt = f"""
You are a professional Pakistani nutritionist.
Generate a 1-day diet plan using common Pakistani foods.

User Targets:
Calories: {round(calories)} | Protein: {round(protein)}g | Fats: {round(fats)}g | Carbs: {round(carbs)}g
Goal: {goal} | Budget: {budget}

Return ONLY valid JSON array in this exact format:
[
  {{
    "Meal": "Breakfast",
    "Food Items": "2 eggs + 1 roti",
    "Calories": 400,
    "Protein (g)": 25,
    "Fats (g)": 15,
    "Carbs (g)": 40,
    "Total Macros": "Protein:25g Fat:15g Carbs:40g"
  }}
]
Include Breakfast, Lunch and Dinner. Do not write anything else.
"""
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": "You are a professional nutritionist."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.5
        )

        text = response.choices[0].message.content
        diet_data = safe_json_parse(text)

        if not diet_data:
            return gr.update(visible=True), None, None # Keep section visible even on fail

        df = pd.DataFrame(diet_data)
        csv_path = "diet_plan.csv"
        df.to_csv(csv_path, index=False)

        return gr.update(visible=True), df, csv_path

    except Exception as e:
        print("ERROR:", e)
        return gr.update(visible=True), None, None

def evaluate_goal(bmi, goal):
    if bmi < 18.5:
        correct_goal = "Increase weight"
    elif bmi < 25:
        correct_goal = "Maintain weight"
    else:
        correct_goal = "Decrease weight"

    if goal == correct_goal:
        return f"✅ **Excellent choice!** {goal} is appropriate for your body."
    else:
        return f"⚠️ **Note:** Based on your BMI, a healthier aim might be to {correct_goal}."

def calculate_health_report(gender, age, ft, inch, weight, activity, goal):
    if None in [gender, age, ft, weight, activity, goal]:
        return gr.update(visible=True), gr.update(visible=False), "⚠️ Please fill all fields.", None, None, None, None, None

    inch = inch or 0
    total_inches = (ft * 12) + inch
    height_cm = total_inches * 2.54
    height_m = height_cm / 100

    bmi = weight / (height_m ** 2)
    goal_msg = evaluate_goal(bmi, goal)

    if gender == "Male":
        bmr = 10 * weight + 6.25 * height_cm - 5 * age + 5
    else:
        bmr = 10 * weight + 6.25 * height_cm - 5 * age - 161

    activity_map = {
        "Sedentary": 1.2,
        "Lightly Active": 1.375,
        "Moderately Active": 1.55,
        "Very Active": 1.725,
        "Extra Active": 1.9
    }

    level = activity.split(" - ")[0]
    tdee = bmr * activity_map[level]

    if goal == "Decrease weight":
        target_cal = tdee - 500
    elif goal == "Increase weight":
        target_cal = tdee + 500
    else:
        target_cal = tdee

    protein = (target_cal * 0.25) / 4
    fats = (target_cal * 0.30) / 9
    carbs = (target_cal * 0.45) / 4

    report = f"""
<div class="report-card">
    <h3>📊 Your Personal Health Report</h3>
    <p><strong>BMI:</strong> {round(bmi,1)}</p>
    <p>{goal_msg}</p>
    <hr>
    <p><strong>BMR:</strong> {round(bmr)} kcal/day</p>
    <p><strong>TDEE (Maintenance):</strong> {round(tdee)} kcal/day</p>
    <p style="font-size: 1.2em; color: #047857;"><strong>🎯 Target Calories:</strong> {round(target_cal)} kcal/day</p>
    <br>
    <h4>Macronutrient Breakdown:</h4>
    <ul>
        <li>🍗 <strong>Protein:</strong> {round(protein)}g</li>
        <li>🥑 <strong>Fats:</strong> {round(fats)}g</li>
        <li>🍚 <strong>Carbs:</strong> {round(carbs)}g</li>
    </ul>
</div>
"""
    return gr.update(visible=False), gr.update(visible=True), report, target_cal, protein, fats, carbs, goal

# ----------------------------------
# Enhanced UI / UX Setup
# ----------------------------------
custom_theme = gr.themes.Soft(
    primary_hue="emerald",
    secondary_hue="teal",
    font=[gr.themes.GoogleFont("Poppins"), "sans-serif"]
).set(
    body_text_size="*text_lg",
    block_title_text_size="*text_xl",
    block_label_text_size="*text_lg",
    button_large_text_size="*text_lg",
)

custom_css = """
.gradio-container { font-family: 'Poppins', sans-serif; }
h1, h2, h3, h4 { color: #064e3b !important; }
.step-header { text-align: center; padding: 25px; background: #d1fae5; border-radius: 12px; margin-bottom: 25px; box-shadow: 0 4px 6px rgba(0,0,0,0.05); }
.report-card { background: #f3f4f6; padding: 20px; border-radius: 10px; border-left: 5px solid #10b981; }
.nav-btn { font-weight: bold !important; font-size: 18px !important; padding: 15px !important; }
"""

with gr.Blocks(theme=custom_theme, css=custom_css) as app:

    # State Variables
    cal_state = gr.State()
    p_state = gr.State()
    f_state = gr.State()
    c_state = gr.State()
    g_state = gr.State()

    # --- STEP 1: Basic Info ---
    with gr.Column(visible=True) as section1:
        gr.HTML("<div class='step-header'><h2>🌿 HealthyMe | Step 1: Basic Information</h2><p>Let's start by getting to know your body metrics.</p></div>")

        with gr.Row():
            gender_input = gr.Dropdown(["Male", "Female"], label="Gender", value="Male")
            age_input = gr.Number(label="Age (Years)", value=25)

        with gr.Row():
            height_ft_input = gr.Number(label="Height (Feet)", value=5)
            height_in_input = gr.Number(label="Height (Inches)", value=8)
            weight_input = gr.Number(label="Weight (kg)", value=70)

        btn1 = gr.Button("Next Step ➡️", variant="primary", elem_classes="nav-btn")

    # --- STEP 2: Activity & Goals ---
    with gr.Column(visible=False) as section2:
        gr.HTML("<div class='step-header'><h2>🏃‍♂️ HealthyMe | Step 2: Lifestyle & Goals</h2><p>Tell us about your routine and what you want to achieve.</p></div>")

        activity_input = gr.Radio([
            "Sedentary - Little to no exercise",
            "Lightly Active - Light exercise 1-3 days/week",
            "Moderately Active - Moderate exercise 3-5 days/week",
            "Very Active - Hard exercise 6-7 days/week",
            "Extra Active - Physical job or intense training"
        ], label="Activity Level", value="Moderately Active - Moderate exercise 3-5 days/week")

        goal_input = gr.Radio(["Maintain weight", "Decrease weight", "Increase weight"], label="Primary Goal", value="Decrease weight")

        with gr.Row():
            back2 = gr.Button("⬅️ Back", elem_classes="nav-btn")
            submit2 = gr.Button("Analyze Health 🔍", variant="primary", elem_classes="nav-btn")

    # --- STEP 3: Report & Budget ---
    with gr.Column(visible=False) as section3:
        gr.HTML("<div class='step-header'><h2>📋 HealthyMe | Step 3: Your Health Report</h2><p>Review your metrics and set your meal budget.</p></div>")

        output_text = gr.HTML()

        gr.Markdown("### 🍲 Customize Your Diet Plan")
        budget_input = gr.Radio(["Low Budget", "Medium Budget", "High Budget"], label="Grocery Budget", value="Medium Budget")

        with gr.Row():
            back3 = gr.Button("⬅️ Back to Goals", elem_classes="nav-btn")
            btn_get_diet = gr.Button("Generate AI Diet Plan 🍽️", variant="primary", elem_classes="nav-btn")

    # --- STEP 4: Results ---
    with gr.Column(visible=False) as section4:
        gr.HTML("<div class='step-header'><h2>🥗 HealthyMe | Your AI Diet Plan</h2><p>Here is your personalized daily meal plan.</p></div>")

        diet_table = gr.Dataframe(interactive=False)
        download_file = gr.File(label="Download Plan as CSV")

        restart = gr.Button("🔄 Start Over", variant="secondary", elem_classes="nav-btn")

    # --- Event Listeners for Wizard Flow ---

    # Step 1 -> Step 2
    btn1.click(lambda: (gr.update(visible=False), gr.update(visible=True)), outputs=[section1, section2])

    # Step 2 -> Step 1
    back2.click(lambda: (gr.update(visible=False), gr.update(visible=True)), outputs=[section2, section1])

    # Step 2 -> Step 3
    submit2.click(
        calculate_health_report,
        inputs=[gender_input, age_input, height_ft_input, height_in_input, weight_input, activity_input, goal_input],
        outputs=[section2, section3, output_text, cal_state, p_state, f_state, c_state, g_state]
    )

    # Step 3 -> Step 2
    back3.click(lambda: (gr.update(visible=False), gr.update(visible=True)), outputs=[section3, section2])

    # Step 3 -> Step 4
    def trigger_diet_generation():
        # Transition UI to Step 4 instantly, then generation happens
        return gr.update(visible=False), gr.update(visible=True)

    btn_get_diet.click(
        trigger_diet_generation,
        outputs=[section3, section4]
    ).then(
        generate_ai_diet_plan,
        inputs=[cal_state, p_state, f_state, c_state, g_state, budget_input],
        outputs=[section4, diet_table, download_file]
    )

    # Step 4 -> Step 1 (Reset)
    restart.click(
        lambda: (gr.update(visible=True), gr.update(visible=False), None, None),
        outputs=[section1, section4, diet_table, download_file]
    )

if __name__ == "__main__":
    app.launch(debug=True)

/tmp/ipython-input-1082392581.py:182: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=custom_theme, css=custom_css) as app:
/tmp/ipython-input-1082392581.py:182: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=custom_theme, css=custom_css) as app:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e71e65c927a6b2f933.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
